In [1]:
import pandas as pd

In [2]:
data_source_path = r"C:\Users\venuv\Documents\ai developer\SentimentAnalysis\archive\imbd_reviews.csv".replace("\\", "/")
print(data_source_path)

C:/Users/venuv/Documents/ai developer/SentimentAnalysis/archive/imbd_reviews.csv


In [3]:
df = pd.read_csv(data_source_path)
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [4]:
#remove all the html tags from the review column 
import re
modified_reviews = df["review"].apply(lambda x: re.sub("<.*?>", "", x))

In [5]:
modified_reviews[modified_reviews.apply(lambda x: x.strip()=="")] #checking for empty spaces
modified_reviews[modified_reviews.isna()] #checking for nan values

Series([], Name: review, dtype: object)

In [6]:
#data transformation
modified_reviews = modified_reviews.str.lower()\
.str.replace("[^\w\s]+", "", regex=True)\
.str.replace(r"\s+", " ")\
.str.strip()\
.str.split()
modified_reviews

0        [one, of, the, other, reviewers, has, mentione...
1        [a, wonderful, little, production, the, filmin...
2        [i, thought, this, was, a, wonderful, way, to,...
3        [basically, theres, a, family, where, a, littl...
4        [petter, matteis, love, in, the, time, of, mon...
                               ...                        
49995    [i, thought, this, movie, did, a, down, right,...
49996    [bad, plot, bad, dialogue, bad, acting, idioti...
49997    [i, am, a, catholic, taught, in, parochial, el...
49998    [im, going, to, have, to, disagree, with, the,...
49999    [no, one, expects, the, star, trek, movies, to...
Name: review, Length: 50000, dtype: object

In [7]:
df["review"] = modified_reviews

In [8]:
df["sentiment"] = df["sentiment"].replace("positive", 1).replace("negative", 0)

C:\Users\venuv\AppData\Local\Temp\ipykernel_10544\3780438693.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["sentiment"] = df["sentiment"].replace("positive", 1).replace("negative", 0)


In [9]:
from collections import Counter
wordCounts = Counter()
for tokens in df["review"]:
    wordCounts.update(tokens)
vocab_size = 20000
commonWords = wordCounts.most_common(vocab_size)


In [10]:
vocab = {}
index = 0
for token, _ in commonWords:
    if token not in vocab:
        vocab[token] = index 
        index+=1

In [11]:
indexedVectors = []
for tokens in df["review"]:
    vector = []
    for token in tokens:
        if(vocab.get(token, 0)!=0):
            vector.append(vocab.get(token))
    if(len(vector)==0):
        vector = [-1]
    indexedVectors.append(vector)

In [12]:
import math
def tanh(x):
    return math.tanh(x)

def sigmoid(x):
    return 1/(1+math.exp(-x))

In [14]:
#initialize weights
totalWordsCount = len(vocab)
embbeding_size = 50
hidden_dimensions_size = 50
totalSize = embbeding_size + hidden_dimensions_size

In [16]:
import random
embeedingMatrix = [[random.uniform(-0.5, 0.5) for _ in range(embbeding_size)] for _ in range(totalWordsCount)]
Wf = [[random.uniform(-0.5, 0.5) for _ in range(totalSize)] for _ in range(hidden_dimensions_size)]
Bf = [random.uniform(-0.5,0.5) for _ in range(hidden_dimensions_size)]
Wi = [[random.uniform(-0.5, 0.5) for _ in range(totalSize)] for _ in range(hidden_dimensions_size)]
Bi = [random.uniform(-0.5,0.5) for _ in range(hidden_dimensions_size)]
Wc = [[random.uniform(-0.5, 0.5) for _ in range(totalSize)] for _ in range(hidden_dimensions_size)]
Bc = [random.uniform(-0.5,0.5) for _ in range(hidden_dimensions_size)]
Wo = [[random.uniform(-0.5, 0.5) for _ in range(totalSize)] for _ in range(hidden_dimensions_size)]
Bo = [random.uniform(-0.5,0.5) for _ in range(hidden_dimensions_size)]


In [ ]:
learning_rate = 0.001
epochs = 100

def dsigmoid(x):
    return x * (1 - x)

def dtanh(x):
    return 1 - x*x

def modelTraining():

    global Wf, Wi, Wc, Wo
    global Bf, Bi, Bc, Bo
    global Wy, By

    for epoch in range(epochs):

        total_loss = 0

        for index in range(len(indexedVectors)):

            tokens = indexedVectors[index]
            target = df["sentiment"][index]

            # -----------------------------
            # FORWARD PASS STORAGE
            # -----------------------------

            hs = []
            cs = []

            fs = []
            is_ = []
            cs_tilde = []
            os = []

            xs = []

            h = [0.0] * hidden_dimensions_size
            c = [0.0] * hidden_dimensions_size

            hs.append(h[:])
            cs.append(c[:])

            # -----------------------------
            # FORWARD PASS
            # -----------------------------

            for token in tokens:

                x_t = embeedingMatrix[token]

                z = h + x_t

                f = [0.0] * hidden_dimensions_size
                i_gate = [0.0] * hidden_dimensions_size
                c_tilde = [0.0] * hidden_dimensions_size
                o = [0.0] * hidden_dimensions_size

                for j in range(hidden_dimensions_size):

                    f[j] = sigmoid(
                        sum(Wf[j][k] * z[k] for k in range(len(z))) + Bf[j]
                    )

                    i_gate[j] = sigmoid(
                        sum(Wi[j][k] * z[k] for k in range(len(z))) + Bi[j]
                    )

                    c_tilde[j] = tanh(
                        sum(Wc[j][k] * z[k] for k in range(len(z))) + Bc[j]
                    )

                    o[j] = sigmoid(
                        sum(Wo[j][k] * z[k] for k in range(len(z))) + Bo[j]
                    )

                # cell update
                new_c = [0.0] * hidden_dimensions_size

                for j in range(hidden_dimensions_size):
                    new_c[j] = (
                        f[j] * c[j]
                        + i_gate[j] * c_tilde[j]
                    )

                # hidden update
                new_h = [0.0] * hidden_dimensions_size

                for j in range(hidden_dimensions_size):
                    new_h[j] = o[j] * tanh(new_c[j])

                h = new_h
                c = new_c

                # store values
                xs.append(z)

                fs.append(f)
                is_.append(i_gate)
                cs_tilde.append(c_tilde)
                os.append(o)

                hs.append(h[:])
                cs.append(c[:])

            # -----------------------------
            # OUTPUT LAYER
            # -----------------------------

            y_raw = sum(
                Wy[j] * h[j]
                for j in range(hidden_dimensions_size)
            ) + By

            y_pred = sigmoid(y_raw)

            # binary cross entropy loss
            loss = -(
                target * math.log(y_pred + 1e-8)
                + (1-target) * math.log(1-y_pred + 1e-8)
            )

            total_loss += loss

            # -----------------------------
            # OUTPUT GRADIENT
            # -----------------------------

            dy = y_pred - target

            dWy = [0.0] * hidden_dimensions_size

            for j in range(hidden_dimensions_size):
                dWy[j] = dy * h[j]

            dBy = dy

            dh_next = [0.0] * hidden_dimensions_size
            dc_next = [0.0] * hidden_dimensions_size

            for j in range(hidden_dimensions_size):
                dh_next[j] = dy * Wy[j]

            # -----------------------------
            # INITIALIZE GRADIENTS
            # -----------------------------

            dWf = [
                [0.0] * totalSize
                for _ in range(hidden_dimensions_size)
            ]

            dWi = [
                [0.0] * totalSize
                for _ in range(hidden_dimensions_size)
            ]

            dWc = [
                [0.0] * totalSize
                for _ in range(hidden_dimensions_size)
            ]

            dWo = [
                [0.0] * totalSize
                for _ in range(hidden_dimensions_size)
            ]

            dBf = [0.0] * hidden_dimensions_size
            dBi = [0.0] * hidden_dimensions_size
            dBc = [0.0] * hidden_dimensions_size
            dBo = [0.0] * hidden_dimensions_size

            # -----------------------------
            # BACKPROP THROUGH TIME
            # -----------------------------

            for t in reversed(range(len(tokens))):

                h = hs[t+1]
                h_prev = hs[t]

                c = cs[t+1]
                c_prev = cs[t]

                f = fs[t]
                i_gate = is_[t]
                c_tilde = cs_tilde[t]
                o = os[t]

                z = xs[t]

                # --------------------------------
                # dh
                # --------------------------------

                dh = dh_next[:]

                # --------------------------------
                # output gate gradient
                # --------------------------------

                do = [0.0] * hidden_dimensions_size

                for j in range(hidden_dimensions_size):
                    do[j] = (
                        dh[j]
                        * tanh(c[j])
                        * dsigmoid(o[j])
                    )

                # --------------------------------
                # cell gradient
                # --------------------------------

                dc = [0.0] * hidden_dimensions_size

                for j in range(hidden_dimensions_size):

                    dc[j] = (
                        dh[j]
                        * o[j]
                        * dtanh(tanh(c[j]))
                        + dc_next[j]
                    )

                # --------------------------------
                # forget gate gradient
                # --------------------------------

                df = [0.0] * hidden_dimensions_size

                for j in range(hidden_dimensions_size):
                    df[j] = (
                        dc[j]
                        * c_prev[j]
                        * dsigmoid(f[j])
                    )

                # --------------------------------
                # input gate gradient
                # --------------------------------

                di = [0.0] * hidden_dimensions_size

                for j in range(hidden_dimensions_size):
                    di[j] = (
                        dc[j]
                        * c_tilde[j]
                        * dsigmoid(i_gate[j])
                    )

                # --------------------------------
                # candidate memory gradient
                # --------------------------------

                dc_tilde = [0.0] * hidden_dimensions_size

                for j in range(hidden_dimensions_size):
                    dc_tilde[j] = (
                        dc[j]
                        * i_gate[j]
                        * dtanh(c_tilde[j])
                    )

                # --------------------------------
                # weight gradients
                # --------------------------------

                for j in range(hidden_dimensions_size):

                    for k in range(totalSize):

                        dWf[j][k] += df[j] * z[k]
                        dWi[j][k] += di[j] * z[k]
                        dWc[j][k] += dc_tilde[j] * z[k]
                        dWo[j][k] += do[j] * z[k]

                    dBf[j] += df[j]
                    dBi[j] += di[j]
                    dBc[j] += dc_tilde[j]
                    dBo[j] += do[j]

                # --------------------------------
                # dz gradient
                # --------------------------------

                dz = [0.0] * totalSize

                for k in range(totalSize):

                    for j in range(hidden_dimensions_size):

                        dz[k] += (
                            Wf[j][k] * df[j]
                            + Wi[j][k] * di[j]
                            + Wc[j][k] * dc_tilde[j]
                            + Wo[j][k] * do[j]
                        )

                # split dz
                dh_next = dz[:hidden_dimensions_size]

                # next cell gradient
                for j in range(hidden_dimensions_size):
                    dc_next[j] = dc[j] * f[j]

            # -----------------------------
            # UPDATE WEIGHTS
            # -----------------------------

            for j in range(hidden_dimensions_size):

                for k in range(totalSize):

                    Wf[j][k] -= learning_rate * dWf[j][k]
                    Wi[j][k] -= learning_rate * dWi[j][k]
                    Wc[j][k] -= learning_rate * dWc[j][k]
                    Wo[j][k] -= learning_rate * dWo[j][k]

                Bf[j] -= learning_rate * dBf[j]
                Bi[j] -= learning_rate * dBi[j]
                Bc[j] -= learning_rate * dBc[j]
                Bo[j] -= learning_rate * dBo[j]

            for j in range(hidden_dimensions_size):
                Wy[j] -= learning_rate * dWy[j]

            By -= learning_rate * dBy

        print("Epoch:", epoch, "Loss:", total_loss)

In [ ]:
def modelTraining():
    for tokens in indexedVectors:

        h = [0.0]*hidden_dimensions_size
        c = [0.0]*hidden_dimensions_size

        for token in tokens:

            x_t = embeedingMatrix[token]
            x = h + x_t   # concatenate

            f = [0.0]*hidden_dimensions_size
            i_gate = [0.0]*hidden_dimensions_size
            c_tilde = [0.0]*hidden_dimensions_size
            o = [0.0]*hidden_dimensions_size

            # compute gates
            for j in range(hidden_dimensions_size):

                f[j] = sigmoid(sum(Wf[j][k] * x[k] for k in range(len(x))) + Bf[j])
                i_gate[j] = sigmoid(sum(Wi[j][k] * x[k] for k in range(len(x))) + Bi[j])
                c_tilde[j] = tanh(sum(Wc[j][k] * x[k] for k in range(len(x))) + Bc[j])
                o[j] = sigmoid(sum(Wo[j][k] * x[k] for k in range(len(x))) + Bo[j])

            # update cell state
            for j in range(hidden_dimensions_size):
                c[j] = f[j]*c[j] + i_gate[j]*c_tilde[j]

            # update hidden state
            for j in range(hidden_dimensions_size):
                h[j] = o[j] * tanh(c[j])